# Fantasy Football Weekly Projections (Half-PPR)

Generates weekly half-PPR fantasy projections for QB, RB, WR, TE using per-position XGBoost models.
Also outputs per-stat prop columns (pass yds, rush yds, receptions, rec yds) for use in sports betting.

Run via **papermill** from the project root:
```bash
papermill fantasy/predict_fantasy.ipynb /tmp/out.ipynb                                      # auto-detect week
papermill fantasy/predict_fantasy.ipynb /tmp/out.ipynb -p TARGET_SEASON 2025 -p TARGET_WEEK 14
```
Output saved to `fantasy/projections/projections_{season}_week{week}.csv`.

## Parameters

In [10]:
TARGET_SEASON = 2025
TARGET_WEEK   = 10  # None = auto-detect next unplayed week
POS_FILTER    = None  # None | "QB" | "RB" | "WR" | "TE"

## Setup — Imports & Config

In [18]:
import warnings
import joblib
import numpy as np
import pandas as pd
import nflreadpy as nfl
from pathlib import Path
import os

warnings.filterwarnings("ignore")

# Works whether kernel starts from project root or from fantasy/ directly
_cwd         = Path.cwd()
_DIR         = _cwd if _cwd.name == "fantasy" else _cwd / "fantasy"
FEATURES_CSV = _DIR / "features_dataset.csv"
MODEL_DIR    = _DIR / "models"
POSITIONS    = ["QB", "RB", "WR", "TE"]

INJURY_MAP   = {"Out": 0.0, "Doubtful": 0.1, "Questionable": 0.5, "Probable": 0.75}
PRACTICE_MAP = {"Did Not Participate In Practice": 0.0, "Limited Participation in Practice": 0.5, "Full Participation in Practice": 1.0}

TURF_SURFACES = {"astroturf", "fieldturf", "turf", "matrixturf", "sportturf", "astroplay", "a_turf"}

## Step 1 — Load Models

Loads the four main position models (`models/{pos}_model.pkl`) plus eight per-stat prop models:

| Model file | Predicts | 2025 MAE |
|---|---|---|
| `qb_pass_yards_model.pkl` | QB passing yards | 72.5 yds |
| `qb_rush_yards_model.pkl` | QB rushing yards | 12.3 yds |
| `rb_rush_yards_model.pkl` | RB rushing yards | 21.1 yds |
| `rb_rec_yards_model.pkl` | RB receiving yards | 10.3 yds |
| `wr_receptions_model.pkl` | WR receptions | 1.4 rec |
| `wr_rec_yards_model.pkl` | WR receiving yards | 21.2 yds |
| `te_receptions_model.pkl` | TE receptions | 1.3 rec |
| `te_rec_yards_model.pkl` | TE receiving yards | 15.5 yds |

QB models use the reduced QB feature set (receiving cols excluded). All others use the full feature set.

In [19]:
# â”€â”€ Load models â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
models = {}
for pos in POSITIONS:
    saved = joblib.load(MODEL_DIR / f"{pos.lower()}_model.pkl")
    models[pos] = saved
print("Models loaded:", {p: len(m["feature_cols"]) for p, m in models.items()})

# ── QB per-stat models ────────────────────────────────────────────
QB_STAT_NAMES  = ["pass_yards", "rush_yards"]
QB_STAT_MODELS = {}
for stat in QB_STAT_NAMES:
    pkl_path = MODEL_DIR / f"qb_{stat}_model.pkl"
    if pkl_path.exists():
        QB_STAT_MODELS[stat] = joblib.load(pkl_path)
print("QB stat models loaded:", list(QB_STAT_MODELS.keys()))

# ── RB per-stat models ────────────────────────────────────────────
RB_STAT_NAMES  = ["rush_yards", "rec_yards"]
RB_STAT_MODELS = {}
for stat in RB_STAT_NAMES:
    pkl_path = MODEL_DIR / f"rb_{stat}_model.pkl"
    if pkl_path.exists():
        RB_STAT_MODELS[stat] = joblib.load(pkl_path)
print("RB stat models loaded:", list(RB_STAT_MODELS.keys()))

# ── WR per-stat models ────────────────────────────────────────────
WR_STAT_NAMES  = ["receptions", "rec_yards"]
WR_STAT_MODELS = {}
for stat in WR_STAT_NAMES:
    pkl_path = MODEL_DIR / f"wr_{stat}_model.pkl"
    if pkl_path.exists():
        WR_STAT_MODELS[stat] = joblib.load(pkl_path)
print("WR stat models loaded:", list(WR_STAT_MODELS.keys()))

# ── TE per-stat models ────────────────────────────────────────────
TE_STAT_NAMES  = ["receptions", "rec_yards"]
TE_STAT_MODELS = {}
for stat in TE_STAT_NAMES:
    pkl_path = MODEL_DIR / f"te_{stat}_model.pkl"
    if pkl_path.exists():
        TE_STAT_MODELS[stat] = joblib.load(pkl_path)
print("TE stat models loaded:", list(TE_STAT_MODELS.keys()))

# â”€â”€ Determine target season / week â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

Models loaded: {'QB': 61, 'RB': 84, 'WR': 84, 'TE': 84}
QB stat models loaded: ['pass_yards', 'rush_yards']
RB stat models loaded: ['rush_yards', 'rec_yards']
WR stat models loaded: ['receptions', 'rec_yards']
TE stat models loaded: ['receptions', 'rec_yards']


## Step 2 — Detect Target Week

In [20]:
def detect_week(season):
    raw   = nfl.load_schedules([season])
    sched = raw.to_pandas() if hasattr(raw, "to_pandas") else pd.DataFrame(raw)
    reg   = sched[(sched["season"] == season) & (sched["game_type"] == "REG")]
    future = reg[reg["result"].isna()]
    return int(future["week"].min()) if not future.empty else None


if TARGET_WEEK is None:
    TARGET_WEEK = detect_week(TARGET_SEASON)
if TARGET_WEEK is None:
    raise ValueError(f"Season {TARGET_SEASON} complete â€” no upcoming games.")

print(f"\nProjecting Season {TARGET_SEASON}  Week {TARGET_WEEK}"
      + (f"  [{POS_FILTER}]" if POS_FILTER else "  [all positions]"))


Projecting Season 2025  Week 10  [all positions]


## Step 3 — Upcoming Schedule

In [21]:
# â”€â”€ Load upcoming schedule â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
raw_sched = nfl.load_schedules([TARGET_SEASON])
schedule  = raw_sched.to_pandas() if hasattr(raw_sched, "to_pandas") else pd.DataFrame(raw_sched)
schedule["season"] = schedule["season"].astype(int)
schedule["week"]   = schedule["week"].astype(int)

upcoming = schedule[
    (schedule["season"] == TARGET_SEASON) &
    (schedule["week"]   == TARGET_WEEK) &
    (schedule["game_type"] == "REG")
].copy()

if upcoming.empty:
    raise ValueError(f"No REG games found for week {TARGET_WEEK}.")

print(f"{len(upcoming)} games found.")

# Build one row per team per game
home = upcoming[["game_id", "home_team", "away_team", "spread_line", "total_line",
                  "roof", "surface", "temp", "wind", "home_rest", "away_rest", "gameday"]].copy()
home.rename(columns={"home_team": "team", "away_team": "opponent_team", "home_rest": "rest"}, inplace=True)
home["is_home"] = 1

away = upcoming[["game_id", "away_team", "home_team", "spread_line", "total_line",
                  "roof", "surface", "temp", "wind", "home_rest", "away_rest", "gameday"]].copy()
away.rename(columns={"away_team": "team", "home_team": "opponent_team", "away_rest": "rest"}, inplace=True)
away["is_home"] = 0

team_ctx = pd.concat([home, away], ignore_index=True)

team_ctx["temp"]  = team_ctx["temp"].fillna(72)
team_ctx["wind"]  = team_ctx["wind"].fillna(0)
team_ctx["is_dome"]  = team_ctx["roof"].isin(["dome", "closed"]).astype(int)
team_ctx["is_turf"]  = team_ctx["surface"].isin(TURF_SURFACES).astype(int)
team_ctx["effective_wind"] = np.where(team_ctx["is_dome"], 0,  team_ctx["wind"])
team_ctx["effective_temp"] = np.where(team_ctx["is_dome"], 72, team_ctx["temp"])
team_ctx["days_rest"] = team_ctx["rest"].fillna(7)

# implied_team_total: home = (total - spread) / 2, away = (total + spread) / 2
# spread_line is home-perspective (negative = home favored), same as features_dataset
team_ctx["implied_team_total"] = np.where(
    team_ctx["is_home"] == 1,
    (team_ctx["total_line"] - team_ctx["spread_line"]) / 2,
    (team_ctx["total_line"] + team_ctx["spread_line"]) / 2,
)

ctx_cols = ["team", "opponent_team", "is_home", "spread_line", "total_line",
            "implied_team_total", "days_rest", "is_dome", "is_turf",
            "effective_wind", "effective_temp", "gameday", "game_id"]

14 games found.


## Step 4 — Player History

In [22]:
# â”€â”€ Load player history â€” each player's latest rolling form â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
hist = pd.read_csv(FEATURES_CSV)
hist["season"] = hist["season"].astype(int)
hist["week"]   = hist["week"].astype(int)

# Most recent completed row per player = current rolling form
latest = (
    hist.sort_values(["player_id", "season", "week"])
    .groupby("player_id").last().reset_index()
)

# Drop players who haven't appeared in the last two seasons (retired/cut)
latest = latest[latest["season"] >= TARGET_SEASON - 1]

# Latest opponent defensive metrics per team (for upcoming opponent lookup)
DEF_COLS = ["def_epa_allowed_roll4", "def_yards_allowed_roll4",
            "def_pass_rate_faced_roll4", "def_red_zone_allowed_roll4"]
opp_def = (
    hist[["opponent_team", "season", "week"] + DEF_COLS]
    .sort_values(["opponent_team", "season", "week"])
    .groupby("opponent_team").last().reset_index()
    [["opponent_team"] + DEF_COLS]
)

# â”€â”€ Join players with upcoming game context â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
active_teams = team_ctx["team"].tolist()
players = latest[latest["team"].isin(active_teams)].copy()
if POS_FILTER:
    players = players[players["position"] == POS_FILTER]

drop_cols = [c for c in ctx_cols + DEF_COLS if c != "team"]
players.drop(columns=drop_cols, errors="ignore", inplace=True)

players = players.merge(team_ctx[ctx_cols], on="team", how="inner")
players = players.merge(opp_def, on="opponent_team", how="left")
print(f"Players matched to upcoming games: {len(players)}")

Players matched to upcoming games: 590


## Step 5 — Injury & Depth Chart Refresh

- **Injuries**: pulls `nfl.load_injuries()` for `TARGET_SEASON`, filters to `TARGET_WEEK`, maps `report_status` → `injury_status_score`. Players not on the report default to 1.0 (healthy). Players with `injury_status_score == 0` (officially Out) are dropped before projecting.
- **Depth charts**: pulls `nfl.load_depth_charts()` and selects the latest snapshot **before** `TARGET_WEEK`’s first game date. This prevents retroactive runs from using post-promotion depth charts for players who weren’t yet starters.

In [23]:
# â”€â”€ Refresh injuries for this week â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
try:
    raw_inj = nfl.load_injuries(seasons=[TARGET_SEASON])
    inj     = raw_inj.to_pandas() if hasattr(raw_inj, "to_pandas") else pd.DataFrame(raw_inj)
    inj_wk  = inj[inj["week"] == TARGET_WEEK].copy()
    inj_wk["injury_status_score"]   = inj_wk["report_status"].map(INJURY_MAP).fillna(1.0)
    inj_wk["practice_status_score"] = inj_wk["practice_status"].map(PRACTICE_MAP).fillna(1.0)
    inj_wk = inj_wk[["gsis_id", "injury_status_score", "practice_status_score"]].rename(
        columns={"gsis_id": "player_id"}
    )
    players.drop(columns=["injury_status_score", "practice_status_score"], errors="ignore", inplace=True)
    players = players.merge(inj_wk, on="player_id", how="left")
    players["injury_status_score"]   = players["injury_status_score"].fillna(1.0)
    players["practice_status_score"] = players["practice_status_score"].fillna(1.0)
    print("Injuries updated.")
except Exception as e:
    print(f"Injuries unavailable ({e}) â€” using last known values")

# ── Refresh depth charts ────────────────────────────────────────────
try:
    raw_dc = nfl.load_depth_charts(seasons=[TARGET_SEASON])
    dc     = raw_dc.to_pandas() if hasattr(raw_dc, "to_pandas") else pd.DataFrame(raw_dc)
    dc["dt"] = pd.to_datetime(dc["dt"], utc=True)
    # Cap snapshot to before the target week's first game so retroactive runs
    # don't pull post-promotion depth charts for players not yet the starter.
    week_cutoff = pd.to_datetime(players["gameday"].min(), utc=True)
    dc_before = dc[dc["dt"] <= week_cutoff]
    latest_dt = dc_before["dt"].max() if not dc_before.empty else dc["dt"].max()
    dc_latest = dc[dc["dt"] == latest_dt].copy()
    # pos_abb is the position (QB/RB/WR/TE), pos_rank is depth (1=starter)
    dc_skill = dc_latest[dc_latest["pos_abb"].isin(["QB", "RB", "WR", "TE"])]
    dc_clean = (
        dc_skill
        .sort_values("pos_rank")
        .drop_duplicates(subset=["gsis_id"], keep="first")
        [["gsis_id", "pos_rank"]]
        .rename(columns={"gsis_id": "player_id", "pos_rank": "depth_chart_position"})
    )
    players.drop(columns=["depth_chart_position"], errors="ignore", inplace=True)
    players = players.merge(dc_clean, on="player_id", how="left")
    players["depth_chart_position"] = players["depth_chart_position"].fillna(2)
    print(f"Depth charts updated from snapshot: {latest_dt.date()}")
except Exception as e:
    print(f"Depth charts unavailable ({e}) — using last known values")
# ── Recompute team rankings from current rolling values ───────────────────────
# Use most recent team-level row from hist so all players on the same team
# share one authoritative value, then rank within this week's matchups.
team_metrics = (
    hist[['team', 'season', 'week', 'off_epa_roll4', 'opp_win_pct_roll4']]
    .drop_duplicates(subset=['team', 'season', 'week'])
    .sort_values(['team', 'season', 'week'])
    .groupby('team').last()
    .reset_index()[['team', 'off_epa_roll4', 'opp_win_pct_roll4']]
    .rename(columns={'off_epa_roll4': '_epa', 'opp_win_pct_roll4': '_sos'})
)
active = team_metrics[team_metrics['team'].isin(players['team'].unique())].copy()
active['off_epa_rank'] = active['_epa'].rank(ascending=False, method='min').astype(int)
active['sos_rank']     = active['_sos'].rank(ascending=False, method='min').astype(int)

players.drop(columns=['off_epa_roll4', 'opp_win_pct_roll4', 'off_epa_rank', 'sos_rank'], errors='ignore', inplace=True)
players = players.merge(active[['team', '_epa', '_sos', 'off_epa_rank', 'sos_rank']], on='team', how='left')
players.rename(columns={'_epa': 'off_epa_roll4', '_sos': 'opp_win_pct_roll4'}, inplace=True)
print(f"Rankings recomputed across {len(active)} teams playing this week.")

# Drop players officially ruled Out
before = len(players)
players = players[players["injury_status_score"] > 0.0]
print(f"Dropped {before - len(players)} players ruled Out.")

Injuries updated.
Depth charts updated from snapshot: 2025-11-05
Rankings recomputed across 28 teams playing this week.
Dropped 22 players ruled Out.


## Step 6 — Generate Projections

Runs the main position model for each position to produce `projected_pts` (half-PPR total).
For positions with per-stat models, also generates prop columns:

| Position | Prop columns added |
|---|---|
| QB | `pred_qb_pass_yards`, `pred_qb_rush_yards` |
| RB | `pred_rush_yards`, `pred_rec_yards` |
| WR | `pred_wr_receptions`, `pred_wr_rec_yards` |
| TE | `pred_te_receptions`, `pred_te_rec_yards` |

All prop predictions are clipped at 0 (no negative yards/receptions).

In [24]:
# ── Score with position models ────────────────────────────────────────────────
all_proj = []

for pos in POSITIONS:
    if POS_FILTER and pos != POS_FILTER:
        continue
    pos_players = players[players["position"] == pos].copy()
    if pos_players.empty:
        continue

    feat_cols = models[pos]["feature_cols"]
    missing   = [c for c in feat_cols if c not in pos_players.columns]
    if missing:
        print(f"  {pos}: {len(missing)} features missing — filling 0: {missing[:5]}")
        for c in missing:
            pos_players[c] = 0

    X = pos_players[feat_cols].fillna(pos_players[feat_cols].median())
    pos_players = pos_players.copy()
    pos_players["projected_pts"] = models[pos]["model"].predict(X).round(2)

    # QB per-stat breakdown
    if pos == "QB" and QB_STAT_MODELS:
        for stat, stat_model in QB_STAT_MODELS.items():
            sc = stat_model["feature_cols"]
            Xs = pos_players[sc].fillna(pos_players[sc].median())
            pos_players[f"pred_qb_{stat}"] = np.clip(stat_model["model"].predict(Xs), 0, None).round(2)

    # RB per-stat breakdown
    if pos == "RB" and RB_STAT_MODELS:
        for stat, stat_model in RB_STAT_MODELS.items():
            sc = stat_model["feature_cols"]
            Xs = pos_players[sc].fillna(pos_players[sc].median())
            pos_players[f"pred_{stat}"] = np.clip(stat_model["model"].predict(Xs), 0, None).round(2)

    # WR per-stat breakdown
    if pos == "WR" and WR_STAT_MODELS:
        for stat, stat_model in WR_STAT_MODELS.items():
            sc = stat_model["feature_cols"]
            Xs = pos_players[sc].fillna(pos_players[sc].median())
            pos_players[f"pred_wr_{stat}"] = np.clip(stat_model["model"].predict(Xs), 0, None).round(2)

    # TE per-stat breakdown
    if pos == "TE" and TE_STAT_MODELS:
        for stat, stat_model in TE_STAT_MODELS.items():
            sc = stat_model["feature_cols"]
            Xs = pos_players[sc].fillna(pos_players[sc].median())
            pos_players[f"pred_te_{stat}"] = np.clip(stat_model["model"].predict(Xs), 0, None).round(2)

    all_proj.append(pos_players)

if not all_proj:
    raise ValueError("No projections generated, check that features_dataset.csv is up to date.")

proj = pd.concat(all_proj, ignore_index=True)

# ── Print + save ──────────────────────────────────────────────────────────────
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 120)

SEP = "=" * 65
print(SEP)
print(f"  Season {TARGET_SEASON}  Week {TARGET_WEEK}  Half-PPR Projections")
print(SEP)

for pos in POSITIONS:
    if POS_FILTER and pos != POS_FILTER:
        continue
    subset = (
        proj[proj["position"] == pos]
        .sort_values("projected_pts", ascending=False)
        .head(20)
        [["player_display_name", "team", "opponent_team", "projected_pts",
          "implied_team_total", "depth_chart_position", "injury_status_score"]]
        .reset_index(drop=True)
    )
    if subset.empty:
        continue
    subset.index += 1
    print()
    print(f"--- {pos} (Top 20) ---")
    print(subset.to_string())

qb_stat_cols = [f"pred_qb_{s}" for s in QB_STAT_NAMES if f"pred_qb_{s}" in proj.columns]
rb_stat_cols = ["pred_rush_yards", "pred_rec_yards"]
wr_stat_cols = [f"pred_wr_{s}" for s in WR_STAT_NAMES if f"pred_wr_{s}" in proj.columns]
te_stat_cols = [f"pred_te_{s}" for s in TE_STAT_NAMES if f"pred_te_{s}" in proj.columns]
out_path = _DIR / "projections" / f"projections_{TARGET_SEASON}_week{TARGET_WEEK:02d}.csv"
os.makedirs(out_path.parent, exist_ok=True)
proj[["player_id", "player_display_name", "position", "team", "opponent_team",
      "gameday", "projected_pts", "implied_team_total",
      "depth_chart_position", "injury_status_score", "is_home",
      "off_epa_roll4", "opp_season_win_pct", "opp_win_pct_roll4",
      "off_epa_rank", "sos_rank"] + qb_stat_cols + rb_stat_cols + wr_stat_cols + te_stat_cols].sort_values(
    ["position", "projected_pts"], ascending=[True, False]
).to_csv(out_path, index=False)
print()
print(f"Saved: {out_path}")


  Season 2025  Week 10  Half-PPR Projections

--- QB (Top 20) ---
   player_display_name team opponent_team  projected_pts  implied_team_total  depth_chart_position  injury_status_score
1          Brock Purdy   SF            LA      24.059999               27.50                   1.0                  0.5
2           Josh Allen  BUF           MIA      22.340000               21.00                   1.0                  1.0
3               Bo Nix  DEN            LV      21.049999               16.50                   1.0                  1.0
4           Drake Maye   NE            TB      19.780001               25.50                   1.0                  1.0
5           Jared Goff  DET           WAS      18.969999               20.50                   1.0                  1.0
6        Lamar Jackson  BAL           MIN      18.969999               22.00                   1.0                  1.0
7          Jalen Hurts  PHI            GB      18.709999               23.50                  

Fantasy Points (Half-PPR)

  RB and TE are genuinely solid. RB at 4.33 MAE and TE at 3.08 are competitive with industry tools — the 2025 holdout results from training (4.49 and 3.26) are holding up in weeks 10–17, which is a good sign the model isn't degrading mid-season. The bias is near-zero for both, meaning it's
  not systematically over or under-projecting. Correlation of 0.63 for RBs is strong.

  WR is acceptable but the top-12 hit rate (31%) is the weak spot. WR is the hardest position to predict in fantasy because usage is so volatile — a wide receiver can be targeted 12 times or 2 times depending on game script and what the defense does. The overall MAE of 3.94 is fine, but only
  getting 4 of 12 top-WRs right per week is below where you'd want to be for lineup decisions.

  QB is the weakest. MAE of 6.26 is decent in absolute terms, but QB fantasy scores are so high-variance (a single garbage-time interception or scramble TD swings 8 points) that correlation of 0.42 is mediocre. The +1.05 bias means it's consistently over-projecting, likely because it projects
   baseline production but doesn't account for blowouts where starters get pulled early.

  ---
  Yard Props (for betting)

  This is actually encouraging:

  - RB Rush Yds (MAE 19.9) — most sportsbooks set lines in 5-yard increments with ~10-yard juice windows. Being within 20 yards on average is usable, especially with near-zero bias
  - RB Rec Yds (MAE 10.7) and TE Rec Yds (MAE 15.3) — these are your strongest props. TE receiving yard lines are typically set around 30-50 yards, so being within 15 is meaningful edge
  - WR Rec Yds (MAE 21.0) — harder to use, same volatility problem as fantasy points
  - QB Pass Yds (MAE 64.8, Bias +14.4) — this is rough. Over/unders on passing yards are typically set within a 20-30 yard range of true expectation, so being off by 65 yards on average with a systematic +14 yard over-projection isn't sharp enough to bet confidently yet. The over-projection
  bias is likely the same blowout/garbage-time issue

  Bottom line: The model is legitimately useful for RB and TE prop betting, and respectable for fantasy lineup decisions at those positions. QB and WR are noisier — use them directionally but don't lean too hard on specific numbers.

## Step 8 — Projection Analysis

Summarizes this week's projections:
- **Distribution** of projected half-PPR points by position (min / 25th / median / 75th / max)
- **Prop stat leaders** by category (rush yards, rec yards, receptions, pass yards)
- **Position scorecards** showing top 10 per position with inline prop stats

Run this cell after Step 6 to inspect the projections before the week is played.

In [25]:
import textwrap

SEP2 = "-" * 65

# ── Distribution summary ─────────────────────────────────────────
print("PROJECTED PTS DISTRIBUTION")
print(SEP2)
dist = (
    proj.groupby("position")["projected_pts"]
    .describe(percentiles=[0.25, 0.5, 0.75])
    [["count", "min", "25%", "50%", "75%", "max", "mean"]]
    .round(1)
)
print(dist.to_string())
print()

# ── Prop stat leaders ────────────────────────────────────────────
print("PROP STAT LEADERS")
print(SEP2)

def top_n(df, col, label, n=5):
    if col not in df.columns:
        return
    rows = df[df[col].notna()].nlargest(n, col)[["player_display_name", "team", col]]
    if rows.empty:
        return
    print(f"{label}:")
    for _, r in rows.iterrows():
        print(f"  {r['player_display_name']:<22} {r['team']:<4} {r[col]:>6.1f}")
    print()

top_n(proj, "pred_qb_pass_yards",  "QB Pass Yards")
top_n(proj, "pred_qb_rush_yards",  "QB Rush Yards")
top_n(proj, "pred_rush_yards",     "RB Rush Yards")
top_n(proj, "pred_rec_yards",      "RB Rec Yards")
top_n(proj, "pred_wr_receptions",  "WR Receptions")
top_n(proj, "pred_wr_rec_yards",   "WR Rec Yards")
top_n(proj, "pred_te_receptions",  "TE Receptions")
top_n(proj, "pred_te_rec_yards",   "TE Rec Yards")

# ── Top 10 per position with inline prop stats ───────────────────
print("TOP 10 PER POSITION (with prop stats)")
print(SEP2)

pos_prop_cols = {
    "QB": [("pred_qb_pass_yards", "PassYds"), ("pred_qb_rush_yards", "RushYds")],
    "RB": [("pred_rush_yards",    "RushYds"), ("pred_rec_yards",    "RecYds")],
    "WR": [("pred_wr_receptions", "Rec"),     ("pred_wr_rec_yards", "RecYds")],
    "TE": [("pred_te_receptions", "Rec"),     ("pred_te_rec_yards", "RecYds")],
}

for pos in ["QB", "RB", "WR", "TE"]:
    if POS_FILTER and pos != POS_FILTER:
        continue
    subset = proj[proj["position"] == pos].nlargest(10, "projected_pts").reset_index(drop=True)
    if subset.empty:
        continue
    prop_defs = pos_prop_cols[pos]
    print(f"{pos}")
    header = f"  {'#':>2}  {'Player':<22} {'Team':<4} {'Opp':<4} {'ProjPts':>7}"
    for _, label in prop_defs:
        if any(col in subset.columns for col, _ in [(_[0], _[1])]):
            header += f"  {label:>7}"
    print(header)
    for i, row in subset.iterrows():
        line = f"  {i+1:>2}  {str(row.get('player_display_name','')):<22} {str(row.get('team','')):<4} {str(row.get('opponent_team','')):<4} {row['projected_pts']:>7.1f}"
        for col, _ in prop_defs:
            if col in subset.columns and not pd.isna(row.get(col)):
                line += f"  {row[col]:>7.1f}"
        print(line)
    print()


PROJECTED PTS DISTRIBUTION
-----------------------------------------------------------------
          count  min  25%   50%   75%   max  mean
position                                         
QB         76.0  1.2  7.0  13.4  16.3  24.1  12.2
RB        143.0  0.4  2.7   4.6   8.1  19.1   5.8
TE        123.0  0.4  1.7   2.6   5.2  12.7   3.7
WR        226.0  0.2  2.2   3.8   6.1  19.2   4.6

PROP STAT LEADERS
-----------------------------------------------------------------
QB Pass Yards:
  Brock Purdy            SF    282.7
  Matthew Stafford       LA    277.2
  Philip Rivers          IND   271.9
  Jared Goff             DET   269.8
  Jordan Love            GB    262.0

QB Rush Yards:
  Anthony Richardson     IND    47.7
  Justin Fields          NYJ    36.4
  Josh Allen             BUF    33.3
  Drake Maye             NE     32.1
  Jalen Hurts            PHI    29.5

RB Rush Yards:
  James Cook             BUF    88.9
  De'Von Achane          MIA    88.1
  Derrick Henry          BAL   